In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 
import subprocess
import optuna

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
df = pd.read_csv('loan.csv', low_memory=False)
print("Shape:", df.shape)
print("\nFirst 5 columns:", df.columns[:5].tolist())
print("\nTarget value counts:")
print(df['loan_status'].value_counts())

Shape: (2260668, 145)

First 5 columns: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv']

Target value counts:
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


In [5]:
# Keep only completed loans with known outcome
keep_status = ['Fully Paid', 'Charged Off', 'Default', 
               'Late (31-120 days)', 'Late (16-30 days)']

df = df[df['loan_status'].isin(keep_status)].copy()

# Binary target: 1 = default, 0 = paid
df['target'] = df['loan_status'].apply(
    lambda x: 0 if x == 'Fully Paid' else 1
)

print("Shape after filtering:", df.shape)
print("\nTarget distribution:")
print(df['target'].value_counts())
print("\nClass balance:", round(df['target'].mean() * 100, 2), "% default")

Shape after filtering: (1329272, 146)

Target distribution:
target
0    1041952
1     287320
Name: count, dtype: int64

Class balance: 21.61 % default


In [7]:
# Check date column
print(df['issue_d'].dtype)
print(df['issue_d'].value_counts().sort_index().head(10))
print("\nDate range:", df['issue_d'].min(), "→", df['issue_d'].max())

object
issue_d
Apr-2008      155
Apr-2009      290
Apr-2010      827
Apr-2011     1563
Apr-2012     3230
Apr-2013     9419
Apr-2014    18117
Apr-2015    31765
Apr-2016    23410
Apr-2017    13015
Name: count, dtype: int64

Date range: Apr-2008 → Sep-2018


In [9]:
# Convert to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# Extract year for splitting
df['year'] = df['issue_d'].dt.year

print("Loans per year:")
print(df.groupby('year')['target'].agg(['count', 'mean']).round(3))

Loans per year:
       count   mean
year               
2007     251  0.179
2008    1562  0.158
2009    4716  0.126
2010   11536  0.129
2011   21721  0.152
2012   53367  0.162
2013  134803  0.156
2014  221902  0.187
2015  375220  0.205
2016  280472  0.258
2017  168699  0.274
2018   55023  0.269


In [11]:
# Temporal split based on actual data distribution
train = df[df['year'] <= 2014].copy()
val   = df[df['year'] == 2015].copy()
test  = df[df['year'] >= 2016].copy()

print(f"Train: {len(train):,} rows | Default rate: {train['target'].mean():.3f}")
print(f"Val:   {len(val):,} rows | Default rate: {val['target'].mean():.3f}")
print(f"Test:  {len(test):,} rows | Default rate: {test['target'].mean():.3f}")

Train: 449,858 rows | Default rate: 0.171
Val:   375,220 rows | Default rate: 0.205
Test:  504,194 rows | Default rate: 0.265


In [12]:
# Calculate missing % for all columns
missing = df.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(2)

# Show top 30 columns with most missing
print("Top 30 columns by missing %:")
print(missing_pct[missing_pct > 0].head(30).to_string())

Top 30 columns by missing %:
id                                            100.00
member_id                                     100.00
url                                           100.00
orig_projected_additional_accrued_interest     99.62
sec_app_mths_since_last_major_derog            99.48
hardship_amount                                99.46
hardship_length                                99.46
hardship_reason                                99.46
hardship_status                                99.46
deferral_term                                  99.46
hardship_start_date                            99.46
hardship_end_date                              99.46
payment_plan_start_date                        99.46
hardship_dpd                                   99.46
hardship_loan_status                           99.46
hardship_payoff_balance_amount                 99.46
hardship_last_payment_amount                   99.46
hardship_type                                  99.46
sec_app_revol_uti

In [15]:
# Drop columns with >40% missing
threshold = 0.4
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with >40% missing")

df = df.drop(columns=cols_to_drop)

# Also drop ID/leakage columns not useful for modeling
drop_cols = ['id', 'member_id', 'loan_status', 'url', 'desc', 
             'title', 'zip_code', 'issue_d', 'year']
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

print(f"Remaining columns: {df.shape[1]}")
print(f"Remaining rows: {df.shape[0]:,}")

Dropping 58 columns with >40% missing
Remaining columns: 84
Remaining rows: 1,329,272


In [16]:
# Redo temporal split on cleaned df
train = df[df.index.isin(train.index)].copy()
val   = df[df.index.isin(val.index)].copy()
test  = df[df.index.isin(test.index)].copy()

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

# Check remaining missing values
remaining_missing = df.isnull().mean()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"\nColumns still with missing values: {len(remaining_missing)}")
print(remaining_missing.head(20).round(3).to_string())

Train: 449,858 | Val: 375,220 | Test: 504,194

Columns still with missing values: 45
mths_since_recent_inq    0.129
num_tl_120dpd_2m         0.088
mo_sin_old_il_acct       0.079
emp_title                0.064
emp_length               0.059
pct_tl_nvr_dlq           0.051
avg_cur_bal              0.051
mo_sin_rcnt_rev_tl_op    0.051
mo_sin_old_rev_tl_op     0.051
num_rev_accts            0.051
num_op_rev_tl            0.051
num_actv_rev_tl          0.051
num_bc_tl                0.051
num_il_tl                0.051
num_actv_bc_tl           0.051
num_rev_tl_bal_gt_0      0.051
num_tl_30dpd             0.051
num_tl_90g_dpd_24m       0.051
num_tl_op_past_12m       0.051
tot_hi_cred_lim          0.051


In [19]:
# Drop emp_title (free text, not useful)
df = df.drop(columns=['emp_title'], errors='ignore')

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Remove target from num_cols
num_cols = [c for c in num_cols if c != 'target']

print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

# Median impute numerical
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Mode impute categorical (emp_length and others)
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Verify no missing values remain
print(f"\nMissing values remaining: {df.isnull().sum().sum()}")

Numerical columns: 65
Categorical columns: 17

Missing values remaining: 0


In [20]:
# Check what categorical columns we have
print("Categorical columns:")
for col in cat_cols:
    print(f"  {col}: {df[col].nunique()} unique values")

Categorical columns:
  term: 2 unique values
  grade: 7 unique values
  sub_grade: 35 unique values
  emp_length: 11 unique values
  home_ownership: 6 unique values
  verification_status: 3 unique values
  pymnt_plan: 2 unique values
  purpose: 14 unique values
  addr_state: 51 unique values
  earliest_cr_line: 738 unique values
  initial_list_status: 2 unique values
  last_pymnt_d: 134 unique values
  last_credit_pull_d: 139 unique values
  application_type: 2 unique values
  hardship_flag: 2 unique values
  disbursement_method: 2 unique values
  debt_settlement_flag: 2 unique values


In [23]:
# Drop high-cardinality date columns and addr_state
drop_high_card = ['earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d', 'addr_state']
df = df.drop(columns=drop_high_card)

# Ordinal encode grade (A=1 best, G=7 worst)
grade_order = {'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7}
df['grade'] = df['grade'].map(grade_order)

# Ordinal encode sub_grade (A1=1 ... G5=35)
sub_grades = [f'{g}{n}' for g in 'ABCDEFG' for n in range(1,6)]
sub_grade_order = {v: i+1 for i, v in enumerate(sub_grades)}
df['sub_grade'] = df['sub_grade'].map(sub_grade_order)

# Ordinal encode emp_length
emp_order = {'< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4,
             '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10}
df['emp_length'] = df['emp_length'].map(emp_order)

# Ordinal encode term
df['term'] = df['term'].str.strip().map({'36 months':36,'60 months':60})

# One-hot encode remaining categoricals
cat_to_encode = ['home_ownership','verification_status','pymnt_plan','purpose',
                 'initial_list_status','application_type','hardship_flag',
                 'disbursement_method','debt_settlement_flag']
df = pd.get_dummies(df, columns=cat_to_encode, drop_first=True)

print(f"Final shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

Final shape: (1329272, 96)
Missing values: 0


In [25]:
# Redo split on fully encoded df
train = df[df.index.isin(train.index)].copy()
val   = df[df.index.isin(val.index)].copy()
test  = df[df.index.isin(test.index)].copy()

# Separate features and target
feature_cols = [c for c in df.columns if c != 'target']

X_train = train[feature_cols]
y_train = train['target']

X_val = val[feature_cols]
y_val = val['target']

X_test = test[feature_cols]
y_test = test['target']

# Scale numerical features
scaler = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_val   = pd.DataFrame(scaler.transform(X_val),       columns=feature_cols)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

print(f"X_train: {X_train.shape} | y_train default rate: {y_train.mean():.3f}")
print(f"X_val:   {X_val.shape}   | y_val default rate:   {y_val.mean():.3f}")
print(f"X_test:  {X_test.shape}  | y_test default rate:  {y_test.mean():.3f}")
print("\nPhase 1 — Data Pipeline COMPLETE")

X_train: (449858, 95) | y_train default rate: 0.171
X_val:   (375220, 95)   | y_val default rate:   0.205
X_test:  (504194, 95)  | y_test default rate:  0.265

Phase 1 — Data Pipeline COMPLETE


In [31]:
y_train_arr = y_train.values
y_val_arr   = y_val.values
y_test_arr  = y_test.values

print("y arrays ready")

y arrays ready


In [33]:
os.makedirs('acrs_data', exist_ok=True)

X_train.to_parquet('acrs_data/X_train.parquet')
X_val.to_parquet('acrs_data/X_val.parquet')
X_test.to_parquet('acrs_data/X_test.parquet')

np.save('acrs_data/y_train.npy', y_train_arr)
np.save('acrs_data/y_val.npy', y_val_arr)
np.save('acrs_data/y_test.npy', y_test_arr)

print("Saved — never reprocess again")

Saved — never reprocess again


In [37]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Quick model WITH leakage (proof)
xgb_leak = XGBClassifier(n_estimators=100, max_depth=5, 
                          eval_metric='auc', random_state=42, n_jobs=-1)
xgb_leak.fit(X_train, y_train_arr)
auc_leak = roc_auc_score(y_test_arr, xgb_leak.predict_proba(X_test)[:,1])
print(f"AUC WITH leakage columns: {auc_leak:.4f}  ← inflated, not real")

AUC WITH leakage columns: 0.9999  ← inflated, not real


In [39]:
# Drop post-outcome leakage columns
leakage_cols = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv',
    'funded_amnt_inv', 'installment'
]
leakage_cols = [c for c in leakage_cols if c in X_train.columns]
print(f"Dropping {len(leakage_cols)} leakage columns")

X_train = X_train.drop(columns=leakage_cols)
X_val   = X_val.drop(columns=leakage_cols)
X_test  = X_test.drop(columns=leakage_cols)

print(f"Clean features: {X_train.shape[1]}")

# Overwrite saved files with clean version
X_train.to_parquet('acrs_data/X_train.parquet')
X_val.to_parquet('acrs_data/X_val.parquet')
X_test.to_parquet('acrs_data/X_test.parquet')

print("Clean data saved — leakage columns permanently removed")

Dropping 12 leakage columns
Clean features: 83
Clean data saved — leakage columns permanently removed
